# 01 — Embed and Block

**Run once. Safe to re-run — every embedding file is skipped if it already exists on Drive.**

What this notebook does:
1. Installs dependencies and mounts Drive
2. Generates L2-normalized float16 embeddings for all 6 source TSV files (train S1/S2/S3 + test S1/S2/S3) using the shared `intfloat/multilingual-e5-base` encoder
3. Saves 12 `.npy` embedding files + 12 ID files to `DATASET_ROOT/output/embeddings/`

Downstream notebooks that depend on these outputs:
- `02_prep_pairs.ipynb` — loads train embeddings, runs FAISS blocking, builds pair tensors
- `04_infer.ipynb` — loads test embeddings, runs FAISS blocking, writes submission

In [ ]:
# ── Colab setup: installs, Drive mount ──────────────────────────────────────
!pip install -q transformers sentencepiece faiss-cpu

import os
try:
    import google.colab
    COLAB = True
except ImportError:
    COLAB = False

# EDIT THIS: folder on Drive containing dataset/ and utils/
BASE_PATH = '/content/drive/MyDrive/Amazon/student_resource 2'

if COLAB and BASE_PATH.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

DATA_DIR = os.path.expanduser(BASE_PATH)
assert os.path.isdir(DATA_DIR), f'Folder not found at {DATA_DIR}'
for sub in ['dataset/train', 'dataset/test']:
    assert os.path.isdir(os.path.join(DATA_DIR, sub)), f'missing {sub} under {DATA_DIR}'
print('Dataset OK at', DATA_DIR)

WORK_ROOT = '/content/er'
os.makedirs(WORK_ROOT, exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if COLAB:
    smi = os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null').read().strip()
    print('GPU:', smi or 'none detected')
print('Colab ready.')

## Section 0 — Configuration

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import re
import gc
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
try:
    COLAB
except NameError:
    COLAB = False
    DATA_DIR = None
    WORK_ROOT = None

if COLAB:
    DATASET_ROOT = Path(DATA_DIR)
    ROOT = Path(WORK_ROOT)
else:
    ROOT = Path.cwd()
    if not (ROOT / 'dataset').exists():
        ROOT = Path('..').resolve()
    DATASET_ROOT = ROOT

TRAIN = DATASET_ROOT / 'dataset/train'
TEST  = DATASET_ROOT / 'dataset/test'
OUT   = ROOT / 'output';   OUT.mkdir(exist_ok=True)

# Embeddings live on Drive so they survive disconnects
EMB = DATASET_ROOT / 'output/embeddings'
EMB.mkdir(parents=True, exist_ok=True)

# ── Model config ─────────────────────────────────────────────────────────────
ENCODER_MODEL    = 'intfloat/multilingual-e5-base'
ENCODER_DIM      = 768
EMBED_BATCH_SIZE = 256
MAX_SEQ_LEN      = 48

is_cuda_available = torch.cuda.is_available()
DEVICE = torch.device('cuda' if is_cuda_available else 'cpu')
print(f'Device: {DEVICE}')

np.random.seed(42)
torch.manual_seed(42)

## Section 1 — Text Preprocessing

In [ ]:
LEGAL_SUFFIXES = {
    'incorporated', 'incorporation', 'corporation', 'company',
    'limited', 'liability', 'private', 'public',
    'inc', 'corp', 'llc', 'ltd', 'llp', 'lp', 'pvt', 'co', 'plc',
    'sarl', 'sas', 'sa', 'srl', 'eurl', 'snc'
}

ADDR_ABBREV = {
    'rd': 'road', 'st': 'street', 'ave': 'avenue', 'blvd': 'boulevard',
    'dr': 'drive', 'ln': 'lane', 'ct': 'court', 'pl': 'place',
    'hwy': 'highway', 'pkwy': 'parkway', 'apt': 'apartment',
    'ste': 'suite', 'n': 'north', 's': 'south', 'e': 'east', 'w': 'west',
}

def preprocess_name(text):
    """Normalize business name: strip legal suffixes, normalize & → and."""
    if not text or pd.isna(text):
        return ''
    text = str(text).lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'[^\w\s+]', ' ', text)
    tokens = [t for t in text.split() if t not in LEGAL_SUFFIXES]
    return ' '.join(tokens).strip() or str(text).lower().strip()

def preprocess_address(text):
    """Normalize business address: expand abbreviations, lowercase."""
    if not text or pd.isna(text):
        return ''
    text = str(text).lower().strip()
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = [ADDR_ABBREV.get(t, t) for t in text.split()]
    return ' '.join(tokens).strip()

def load_tsv_texts(tsv_path):
    """Load entity IDs, business names, and addresses from a TSV file."""
    df = pd.read_csv(tsv_path, sep='\t')
    ids = df['entity_id'].tolist()
    names = df['business_name'].tolist()
    addrs = df['business_address'].tolist()
    return ids, names, addrs

## Section 1B — Shared Encoder

In [ ]:
def embed_with_shared(texts, model, tokenizer,
                      save_path_emb, save_path_ids, ids,
                      batch_size=EMBED_BATCH_SIZE, prefix='passage: '):
    """
    Dense L2-normalized embeddings from the shared multilingual encoder.
    Saves results as float16 .npy files. Skips if save_path_emb already exists.
    """
    if save_path_emb.exists():
        print(f'SKIP: {save_path_emb.name} already exists')
        return

    prefixed = [prefix + t for t in texts]
    all_embeddings = []

    for i in tqdm(range(0, len(prefixed), batch_size), desc=save_path_emb.name):
        batch = prefixed[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=MAX_SEQ_LEN, return_tensors='pt'
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        with torch.no_grad():
            out = model(**encoded)
            mask = encoded['attention_mask'].unsqueeze(-1).float()
            emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_embeddings.append(emb.cpu().float().numpy().astype(np.float16))
        del encoded, out, emb
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    result = np.concatenate(all_embeddings, axis=0)
    np.save(save_path_emb, result)
    np.save(save_path_ids, np.array(ids))
    print(f'Saved {result.shape} → {save_path_emb}')
    del all_embeddings
    gc.collect()

## Section 1C — Encoding Driver

In [ ]:
def encode_all_jobs(jobs, batch_size=EMBED_BATCH_SIZE):
    """Load the encoder ONCE, embed names + addresses for every source file."""
    print(f'Loading encoder: {ENCODER_MODEL} (dense dim {ENCODER_DIM})')
    tokenizer = AutoTokenizer.from_pretrained(ENCODER_MODEL)
    model = AutoModel.from_pretrained(
        ENCODER_MODEL, dtype=torch.float16
    ).to(DEVICE).eval()
    print('Encoder ready — one model produces BOTH name and address embeddings.\n')

    for tsv_path, name_out, addr_out, ids_out in jobs:
        print(f'\n=== Embedding {tsv_path.name} ===')
        ids, names, addrs = load_tsv_texts(tsv_path)
        names = [preprocess_name(n) for n in names]
        addrs = [preprocess_address(a) for a in addrs]

        name_ids_out = Path(str(ids_out).replace('_ids', '_name_ids'))
        addr_ids_out = Path(str(ids_out).replace('_ids', '_addr_ids'))
        embed_with_shared(names, model, tokenizer, name_out, name_ids_out, ids)
        embed_with_shared(addrs, model, tokenizer, addr_out, addr_ids_out, ids)
        del ids, names, addrs
        gc.collect()

    del model, tokenizer
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    print('\nAll embeddings generated.')

## Section 1D — Run Embedding Generation for All 6 Source Files

In [ ]:
EMBED_JOBS = [
    # (source_tsv,               name_emb_out,               addr_emb_out,               ids_out)
    (TRAIN/'train_source1.tsv',  EMB/'train_s1_name.npy',    EMB/'train_s1_addr.npy',    EMB/'train_s1_ids.npy'),
    (TRAIN/'train_source2.tsv',  EMB/'train_s2_name.npy',    EMB/'train_s2_addr.npy',    EMB/'train_s2_ids.npy'),
    (TRAIN/'train_source3.tsv',  EMB/'train_s3_name.npy',    EMB/'train_s3_addr.npy',    EMB/'train_s3_ids.npy'),
    (TEST/'test_source1.tsv',    EMB/'test_s1_name.npy',     EMB/'test_s1_addr.npy',     EMB/'test_s1_ids.npy'),
    (TEST/'test_source2.tsv',    EMB/'test_s2_name.npy',     EMB/'test_s2_addr.npy',     EMB/'test_s2_ids.npy'),
    (TEST/'test_source3.tsv',    EMB/'test_s3_name.npy',     EMB/'test_s3_addr.npy',     EMB/'test_s3_ids.npy'),
]

encode_all_jobs(EMBED_JOBS)

## Done

Embedding files written to `DATASET_ROOT/output/embeddings/`:

```
train_s1_name.npy   train_s1_name_ids.npy
train_s1_addr.npy   train_s1_addr_ids.npy
train_s2_name.npy   train_s2_name_ids.npy
train_s2_addr.npy   train_s2_addr_ids.npy
train_s3_name.npy   train_s3_name_ids.npy
train_s3_addr.npy   train_s3_addr_ids.npy
test_s1_name.npy    test_s1_name_ids.npy
test_s1_addr.npy    test_s1_addr_ids.npy
test_s2_name.npy    test_s2_name_ids.npy
test_s2_addr.npy    test_s2_addr_ids.npy
test_s3_name.npy    test_s3_name_ids.npy
test_s3_addr.npy    test_s3_addr_ids.npy
```

Next step: run `02_prep_pairs.ipynb`.